In [1]:
import numpy as np

**Module** is an abstract class which defines fundamental methods necessary for a training a neural network. You do not need to change anything here, just read the comments.

In [2]:
class Module(object):
    """
    Basically, you can think of a module as of a something (black box)
    which can process `input` data and produce `ouput` data.
    This is like applying a function which is called `forward`:

        output = module.forward(input)

    The module should be able to perform a backward pass: to differentiate the `forward` function.
    More, it should be able to differentiate it if is a part of chain (chain rule).
    The latter implies there is a gradient from previous step of a chain rule.

        gradInput = module.backward(input, gradOutput)
    """
    def __init__ (self):
        self.output = None
        self.gradInput = None
        self.training = True

    def forward(self, input):
        """
        Takes an input object, and computes the corresponding output of the module.
        """
        return self.updateOutput(input)

    def backward(self,input, gradOutput):
        """
        Performs a backpropagation step through the module, with respect to the given input.

        This includes
         - computing a gradient w.r.t. `input` (is needed for further backprop),
         - computing a gradient w.r.t. parameters (to update parameters while optimizing).
        """
        self.updateGradInput(input, gradOutput)
        self.accGradParameters(input, gradOutput)
        return self.gradInput


    def updateOutput(self, input):
        """
        Computes the output using the current parameter set of the class and input.
        This function returns the result which is stored in the `output` field.

        Make sure to both store the data in `output` field and return it.
        """

        # The easiest case:

        # self.output = input
        # return self.output

        pass

    def updateGradInput(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own input.
        This is returned in `gradInput`. Also, the `gradInput` state variable is updated accordingly.

        The shape of `gradInput` is always the same as the shape of `input`.

        Make sure to both store the gradients in `gradInput` field and return it.
        """

        # The easiest case:

        # self.gradInput = gradOutput
        # return self.gradInput

        pass

    def accGradParameters(self, input, gradOutput):
        """
        Computing the gradient of the module with respect to its own parameters.
        No need to override if module has no parameters (e.g. ReLU).
        """
        pass

    def zeroGradParameters(self):
        """
        Zeroes `gradParams` variable if the module has params.
        """
        pass

    def getParameters(self):
        """
        Returns a list with its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def getGradParameters(self):
        """
        Returns a list with gradients with respect to its parameters.
        If the module does not have parameters return empty list.
        """
        return []

    def train(self):
        """
        Sets training mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = True

    def evaluate(self):
        """
        Sets evaluation mode for the module.
        Training and testing behaviour differs for Dropout, BatchNorm.
        """
        self.training = False

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Module"

# Sequential container

**Define** a forward and backward pass procedures.

In [3]:
class Sequential(Module):
    """
         This class implements a container, which processes `input` data sequentially.

         `input` is processed by each module (layer) in self.modules consecutively.
         The resulting array is called `output`.
    """

    def __init__(self):
        super(Sequential, self).__init__()
        self.modules = []

    def add(self, module):
        self.modules.append(module)

    def updateOutput(self, input):
        current_input = input
        for module in self.modules:
            current_input = module.forward(current_input)
        self.output = current_input
        return self.output

    def backward(self, input, gradOutput):
        current_grad = gradOutput

        for i in range(len(self.modules) - 1, -1, -1):
            if i == 0:
                module_input = input
            else:
                module_input = self.modules[i - 1].output

            current_grad = self.modules[i].backward(module_input, current_grad)

        self.gradInput = current_grad
        return self.gradInput

    def zeroGradParameters(self):
        for module in self.modules:
            module.zeroGradParameters()

    def getParameters(self):
        return [x.getParameters() for x in self.modules]

    def getGradParameters(self):
        return [x.getGradParameters() for x in self.modules]

    def __repr__(self):
        string = "".join([str(x) + '\n' for x in self.modules])
        return string

    def __getitem__(self, x):
        return self.modules.__getitem__(x)

    def train(self):
        self.training = True
        for module in self.modules:
            module.train()

    def evaluate(self):
        self.training = False
        for module in self.modules:
            module.evaluate()

# Layers

## 1 (0.2). Linear transform layer
Also known as dense layer, fully-connected layer, FC-layer, InnerProductLayer (in caffe), affine transform
- input:   **`batch_size x n_feats1`**
- output: **`batch_size x n_feats2`**

In [4]:
class Linear(Module):
    """
    A module which applies a linear transformation
    A common name is fully-connected layer, InnerProductLayer in caffe.

    The module should work with 2D input of shape (n_samples, n_feature).
    """
    def __init__(self, n_in, n_out):
        super(Linear, self).__init__()

        # This is a nice initialization
        stdv = 1./np.sqrt(n_in)
        self.W = np.random.uniform(-stdv, stdv, size = (n_out, n_in))
        self.b = np.random.uniform(-stdv, stdv, size = n_out)

        self.gradW = np.zeros_like(self.W)
        self.gradb = np.zeros_like(self.b)

    def updateOutput(self, input):
        self.output = input.dot(self.W.T) + self.b
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.dot(self.W)
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradW += gradOutput.T.dot(input)
        self.gradb += np.sum(gradOutput, axis=0)

    def zeroGradParameters(self):
        self.gradW.fill(0)
        self.gradb.fill(0)

    def getParameters(self):
        return [self.W, self.b]

    def getGradParameters(self):
        return [self.gradW, self.gradb]

    def __repr__(self):
        s = self.W.shape
        q = 'Linear %d -> %d' %(s[1],s[0])
        return q

## 2. (0.2) SoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{softmax}(x)_i = \frac{\exp x_i} {\sum_j \exp x_j}$

Recall that $\text{softmax}(x) == \text{softmax}(x - \text{const})$. It makes possible to avoid computing exp() from large argument.

In [5]:
class SoftMax(Module):
    def __init__(self):
         super(SoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))
        self.output = np.exp(self.output)
        self.output /= np.sum(self.output, axis=1, keepdims=True)
        return self.output

    def updateGradInput(self, input, gradOutput):
        tmp = np.sum(gradOutput * self.output, axis=1, keepdims=True)
        self.gradInput = self.output * (gradOutput - tmp)
        return self.gradInput

    def __repr__(self):
        return "SoftMax"

## 3. (0.2) LogSoftMax
- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

$\text{logsoftmax}(x)_i = \log\text{softmax}(x)_i = x_i - \log {\sum_j \exp x_j}$

The main goal of this layer is to be used in computation of log-likelihood loss.

In [6]:
class LogSoftMax(Module):
    def __init__(self):
         super(LogSoftMax, self).__init__()

    def updateOutput(self, input):
        # start with normalization for numerical stability
        self.output = np.subtract(input, input.max(axis=1, keepdims=True))
        log_sum_exp = np.log(np.sum(np.exp(self.output), axis=1, keepdims=True))
        self.output = self.output - log_sum_exp
        return self.output

    def updateGradInput(self, input, gradOutput):
        softmax = np.exp(self.output)
        sum_grad = np.sum(gradOutput, axis=1, keepdims=True)
        self.gradInput = gradOutput - softmax * sum_grad
        return self.gradInput

    def __repr__(self):
        return "LogSoftMax"

## 4. (0.3) Batch normalization
One of the most significant recent ideas that impacted NNs a lot is [**Batch normalization**](http://arxiv.org/abs/1502.03167). The idea is simple, yet effective: the features should be whitened ($mean = 0$, $std = 1$) all the way through NN. This improves the convergence for deep models letting it train them for days but not weeks. **You are** to implement the first part of the layer: features normalization. The second part (`ChannelwiseScaling` layer) is implemented below.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

The layer should work as follows. While training (`self.training == True`) it transforms input as $$y = \frac{x - \mu}  {\sqrt{\sigma + \epsilon}}$$
where $\mu$ and $\sigma$ - mean and variance of feature values in **batch** and $\epsilon$ is just a small number for numericall stability. Also during training, layer should maintain exponential moving average values for mean and variance:
```
    self.moving_mean = self.moving_mean * alpha + batch_mean * (1 - alpha)
    self.moving_variance = self.moving_variance * alpha + batch_variance * (1 - alpha)
```
During testing (`self.training == False`) the layer normalizes input using moving_mean and moving_variance.

Note that decomposition of batch normalization on normalization itself and channelwise scaling here is just a common **implementation** choice. In general "batch normalization" always assumes normalization + scaling.

In [7]:
class BatchNormalization(Module):
    EPS = 1e-5

    def __init__(self, alpha=0.):
        super(BatchNormalization, self).__init__()
        self.alpha = alpha

        self.moving_mean = None
        self.moving_variance = None
        self.input_shape = None

        self.x_hat = None
        self.std_inv = None
        self.batch_mean = None
        self.batch_var = None

    def _reshape_input(self, input):
        self.input_shape = input.shape
        if len(input.shape) == 2:
            return input
        elif len(input.shape) == 4:
            N, C, H, W = input.shape
            return input.transpose(0, 2, 3, 1).reshape(-1, C)
        else:
            raise ValueError("BatchNormalization supports only 2D or 4D input")

    def _reshape_output(self, output):
        if len(self.input_shape) == 2:
            return output
        elif len(self.input_shape) == 4:
            N, C, H, W = self.input_shape
            return output.reshape(N, H, W, C).transpose(0, 3, 1, 2)
        else:
            raise ValueError("BatchNormalization supports only 2D or 4D input")

    def updateOutput(self, input):
        x = self._reshape_input(input).astype(np.float64)
        n_features = x.shape[1]

        if self.moving_mean is None:
            self.moving_mean = np.zeros(n_features, dtype=np.float64)
            self.moving_variance = np.ones(n_features, dtype=np.float64)

        if self.training:
            self.batch_mean = np.mean(x, axis=0)
            self.batch_var = np.var(x, axis=0, ddof=0)

            self.std_inv = 1.0 / np.sqrt(self.batch_var + self.EPS)
            self.x_hat = (x - self.batch_mean) * self.std_inv

            self.moving_mean = self.alpha * self.moving_mean + (1.0 - self.alpha) * self.batch_mean

            n = x.shape[0]
            unbiased_var = self.batch_var * n / (n - 1) if n > 1 else self.batch_var
            self.moving_variance = self.alpha * self.moving_variance + (1.0 - self.alpha) * unbiased_var

            out = self.x_hat
        else:
            out = (x - self.moving_mean) / np.sqrt(self.moving_variance + self.EPS)

        self.output = self._reshape_output(out.astype(np.float32))
        return self.output

    def updateGradInput(self, input, gradOutput):
        dout = self._reshape_input(gradOutput).astype(np.float64)

        if self.training:
            N = dout.shape[0]

            sum_dout = np.sum(dout, axis=0)
            sum_dout_xhat = np.sum(dout * self.x_hat, axis=0)

            dx = (1.0 / N) * self.std_inv * (
                N * dout - sum_dout - self.x_hat * sum_dout_xhat
            )
        else:
            dx = dout / np.sqrt(self.moving_variance + self.EPS)

        self.gradInput = self._reshape_output(dx.astype(np.float32))
        return self.gradInput

    def __repr__(self):
        return "BatchNormalization"

In [8]:
class ChannelwiseScaling(Module):
    r"""
       Implements linear transform of input y = \gamma * x + \beta
       where \gamma, \beta - learnable vectors of length x.shape[-1]
    """
    def __init__(self, n_out):
        super(ChannelwiseScaling, self).__init__()

        stdv = 1./np.sqrt(n_out)
        self.gamma = np.random.uniform(-stdv, stdv, size=n_out)
        self.beta = np.random.uniform(-stdv, stdv, size=n_out)

        self.gradGamma = np.zeros_like(self.gamma)
        self.gradBeta = np.zeros_like(self.beta)

    def updateOutput(self, input):
        self.output = input * self.gamma + self.beta
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * self.gamma
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        self.gradBeta += np.sum(gradOutput, axis=0)
        self.gradGamma += np.sum(gradOutput * input, axis=0)

    def zeroGradParameters(self):
        self.gradGamma.fill(0)
        self.gradBeta.fill(0)

    def getParameters(self):
        return [self.gamma, self.beta]

    def getGradParameters(self):
        return [self.gradGamma, self.gradBeta]

    def __repr__(self):
        return "ChannelwiseScaling"

Practical notes. If BatchNormalization is placed after a linear transformation layer (including dense layer, convolutions, channelwise scaling) that implements function like `y = weight * x + bias`, than bias adding become useless and could be omitted since its effect will be discarded while batch mean subtraction. If BatchNormalization (followed by `ChannelwiseScaling`) is placed before a layer that propagates scale (including ReLU, LeakyReLU) followed by any linear transformation layer than parameter `gamma` in `ChannelwiseScaling` could be freezed since it could be absorbed into the linear transformation layer.

## 5. (0.3) Dropout
Implement [**dropout**](https://www.cs.toronto.edu/~hinton/absps/JMLRdropout.pdf). The idea and implementation is really simple: just multimply the input by $Bernoulli(p)$ mask. Here $p$ is probability of an element to be zeroed.

This has proven to be an effective technique for regularization and preventing the co-adaptation of neurons.

While training (`self.training == True`) it should sample a mask on each iteration (for every batch), zero out elements and multiply elements by $1 / (1 - p)$. The latter is needed for keeping mean values of features close to mean values which will be in test mode. When testing this module should implement identity transform i.e. `self.output = input`.

- input:   **`batch_size x n_feats`**
- output: **`batch_size x n_feats`**

In [9]:
class Dropout(Module):
    def __init__(self, p=0.5):
        super(Dropout, self).__init__()

        self.p = p
        self.mask = None

    def updateOutput(self, input):
        if self.training:
            self.mask = (np.random.rand(*input.shape) >= self.p) / (1 - self.p)
            self.output = input * self.mask
        else:
            self.output = input
        return self.output

    def updateGradInput(self, input, gradOutput):
        if self.training:
            self.gradInput = gradOutput * self.mask
        else:
            self.gradInput = gradOutput
        return self.gradInput

    def __repr__(self):
        return "Dropout"

#6. (2.0) Conv2d
Implement [**Conv2d**](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html). Use only this list of parameters: (in_channels, out_channels, kernel_size, stride, padding, bias, padding_mode) and fix dilation=1 and groups=1.

In [10]:
class Conv2d(Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride=1, padding='valid', bias=True, padding_mode='zeros'):
        super(Conv2d, self).__init__()

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride if isinstance(stride, tuple) else (stride, stride)
        self.padding = padding
        self.use_bias = bias
        self.padding_mode = padding_mode

        if self.padding not in ['same', 'valid']:
            raise ValueError("Only padding='same' and padding='valid' are supported")

        if self.padding_mode != 'zeros':
            raise ValueError("Only padding_mode='zeros' is supported")

        kh, kw = self.kernel_size
        stdv = 1. / np.sqrt(in_channels * kh * kw)

        self.weight = np.random.uniform(-stdv, stdv, size=(out_channels, in_channels, kh, kw))
        self.gradWeight = np.zeros_like(self.weight)

        if self.use_bias:
            self.bias = np.random.uniform(-stdv, stdv, size=(out_channels,))
            self.gradBias = np.zeros_like(self.bias)
        else:
            self.bias = None
            self.gradBias = None

        self._last_padding = None

    def _pad_input(self, input):
        _, _, H, W = input.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride

        if self.padding == 'valid':
            pad_top = pad_bottom = pad_left = pad_right = 0
        else:  # padding == 'same'
            out_h = int(np.ceil(H / sh))
            out_w = int(np.ceil(W / sw))

            pad_h = max((out_h - 1) * sh + kh - H, 0)
            pad_w = max((out_w - 1) * sw + kw - W, 0)

            pad_top = pad_h // 2
            pad_bottom = pad_h - pad_top
            pad_left = pad_w // 2
            pad_right = pad_w - pad_left

        self._last_padding = (pad_top, pad_bottom, pad_left, pad_right)

        if pad_top == 0 and pad_bottom == 0 and pad_left == 0 and pad_right == 0:
            return input

        return np.pad(
            input,
            ((0, 0), (0, 0), (pad_top, pad_bottom), (pad_left, pad_right)),
            mode='constant',
            constant_values=0
        )

    def updateOutput(self, input):
        batch_size, _, _, _ = input.shape
        out_channels, _, kh, kw = self.weight.shape
        sh, sw = self.stride

        input_padded = self._pad_input(input)
        H_padded, W_padded = input_padded.shape[2], input_padded.shape[3]

        H_out = (H_padded - kh) // sh + 1
        W_out = (W_padded - kw) // sw + 1

        self.output = np.zeros((batch_size, out_channels, H_out, W_out), dtype=input.dtype)

        for n in range(batch_size):
            for oc in range(out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        h_end = h_start + kh
                        w_end = w_start + kw

                        patch = input_padded[n, :, h_start:h_end, w_start:w_end]
                        self.output[n, oc, i, j] = np.sum(patch * self.weight[oc])

                        if self.use_bias:
                            self.output[n, oc, i, j] += self.bias[oc]

        return self.output

    def updateGradInput(self, input, gradOutput):
        batch_size, _, H, W = input.shape
        out_channels, _, kh, kw = self.weight.shape
        sh, sw = self.stride

        input_padded = self._pad_input(input)
        gradInput_padded = np.zeros_like(input_padded)

        H_out, W_out = gradOutput.shape[2], gradOutput.shape[3]

        for n in range(batch_size):
            for oc in range(out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        h_end = h_start + kh
                        w_end = w_start + kw

                        gradInput_padded[n, :, h_start:h_end, w_start:w_end] += (
                            gradOutput[n, oc, i, j] * self.weight[oc]
                        )

        pad_top, pad_bottom, pad_left, pad_right = self._last_padding
        self.gradInput = gradInput_padded[:, :, pad_top:pad_top + H, pad_left:pad_left + W]
        return self.gradInput

    def accGradParameters(self, input, gradOutput):
        batch_size = input.shape[0]
        out_channels, _, kh, kw = self.weight.shape
        sh, sw = self.stride

        input_padded = self._pad_input(input)
        H_out, W_out = gradOutput.shape[2], gradOutput.shape[3]

        for n in range(batch_size):
            for oc in range(out_channels):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        h_end = h_start + kh
                        w_end = w_start + kw

                        patch = input_padded[n, :, h_start:h_end, w_start:w_end]
                        self.gradWeight[oc] += gradOutput[n, oc, i, j] * patch

                        if self.use_bias:
                            self.gradBias[oc] += gradOutput[n, oc, i, j]

    def zeroGradParameters(self):
        self.gradWeight.fill(0)
        if self.use_bias:
            self.gradBias.fill(0)

    def getParameters(self):
        if self.use_bias:
            return [self.weight, self.bias]
        return [self.weight]

    def getGradParameters(self):
        if self.use_bias:
            return [self.gradWeight, self.gradBias]
        return [self.gradWeight]

    def __repr__(self):
        return "Conv2d"

#7. (0.5) Implement [**MaxPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html) and [**AvgPool2d**](https://pytorch.org/docs/stable/generated/torch.nn.AvgPool2d.html). Use only parameters like kernel_size, stride, padding (negative infinity for maxpool and zero for avgpool) and other parameters fixed as in framework.

In [11]:
class MaxPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(MaxPool2d, self).__init__()

        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride if isinstance(stride, tuple) else (stride, stride)
        self.padding = padding if isinstance(padding, tuple) else (padding, padding)

        self.max_indices = None

    def updateOutput(self, input):
        N, C, H, W = input.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        ph, pw = self.padding

        input_padded = np.pad(
            input,
            ((0, 0), (0, 0), (ph, ph), (pw, pw)),
            mode='constant',
            constant_values=-np.inf
        )

        H_out = (H + 2 * ph - kh) // sh + 1
        W_out = (W + 2 * pw - kw) // sw + 1

        self.output = np.zeros((N, C, H_out, W_out))
        self.max_indices = np.zeros((N, C, H_out, W_out, 2), dtype=int)

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        h_end = h_start + kh
                        w_end = w_start + kw

                        window = input_padded[n, c, h_start:h_end, w_start:w_end]
                        max_idx = np.unravel_index(np.argmax(window), window.shape)

                        self.output[n, c, i, j] = window[max_idx]
                        self.max_indices[n, c, i, j] = (h_start + max_idx[0], w_start + max_idx[1])

        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = input.shape
        ph, pw = self.padding

        gradInput_padded = np.zeros((N, C, H + 2 * ph, W + 2 * pw))

        H_out, W_out = gradOutput.shape[2], gradOutput.shape[3]

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_idx, w_idx = self.max_indices[n, c, i, j]
                        gradInput_padded[n, c, h_idx, w_idx] += gradOutput[n, c, i, j]

        if ph == 0 and pw == 0:
            self.gradInput = gradInput_padded
        else:
            self.gradInput = gradInput_padded[:, :, ph:ph + H, pw:pw + W]

        return self.gradInput

    def __repr__(self):
        return "MaxPool2d"

class AvgPool2d(Module):
    def __init__(self, kernel_size, stride, padding):
        super(AvgPool2d, self).__init__()

        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride if isinstance(stride, tuple) else (stride, stride)
        self.padding = padding if isinstance(padding, tuple) else (padding, padding)

    def updateOutput(self, input):
        N, C, H, W = input.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        ph, pw = self.padding

        input_padded = np.pad(
            input,
            ((0, 0), (0, 0), (ph, ph), (pw, pw)),
            mode='constant',
            constant_values=0
        )

        H_out = (H + 2 * ph - kh) // sh + 1
        W_out = (W + 2 * pw - kw) // sw + 1

        self.output = np.zeros((N, C, H_out, W_out))

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        h_end = h_start + kh
                        w_end = w_start + kw

                        window = input_padded[n, c, h_start:h_end, w_start:w_end]
                        self.output[n, c, i, j] = np.mean(window)

        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = input.shape
        kh, kw = self.kernel_size
        sh, sw = self.stride
        ph, pw = self.padding

        gradInput_padded = np.zeros((N, C, H + 2 * ph, W + 2 * pw))

        H_out, W_out = gradOutput.shape[2], gradOutput.shape[3]
        scale = kh * kw

        for n in range(N):
            for c in range(C):
                for i in range(H_out):
                    for j in range(W_out):
                        h_start = i * sh
                        w_start = j * sw
                        h_end = h_start + kh
                        w_end = w_start + kw

                        gradInput_padded[n, c, h_start:h_end, w_start:w_end] += gradOutput[n, c, i, j] / scale

        if ph == 0 and pw == 0:
            self.gradInput = gradInput_padded
        else:
            self.gradInput = gradInput_padded[:, :, ph:ph + H, pw:pw + W]

        return self.gradInput

    def __repr__(self):
        return "AvgPool2d"

#8. (0.3) Implement **GlobalMaxPool2d** and **GlobalAvgPool2d**. They do not have testing and parameters are up to you but they must aggregate information within channels. Write test functions for these layers on your own.

In [12]:
class GlobalMaxPool2d(Module):
    def __init__(self):
        super(GlobalMaxPool2d, self).__init__()
        self.max_indices = None
        self.input_shape = None

    def updateOutput(self, input):
        self.input_shape = input.shape
        N, C, H, W = input.shape

        self.output = np.max(input, axis=(2, 3))
        self.max_indices = np.zeros((N, C, 2), dtype=int)

        for n in range(N):
            for c in range(C):
                idx = np.unravel_index(np.argmax(input[n, c]), (H, W))
                self.max_indices[n, c] = idx

        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = self.input_shape
        self.gradInput = np.zeros((N, C, H, W))

        for n in range(N):
            for c in range(C):
                h, w = self.max_indices[n, c]
                self.gradInput[n, c, h, w] = gradOutput[n, c]

        return self.gradInput

    def __repr__(self):
        return "GlobalMaxPool2d"
    
    
    
class GlobalAvgPool2d(Module):
    def __init__(self):
        super(GlobalAvgPool2d, self).__init__()
        self.input_shape = None

    def updateOutput(self, input):
        self.input_shape = input.shape
        self.output = np.mean(input, axis=(2, 3))
        return self.output

    def updateGradInput(self, input, gradOutput):
        N, C, H, W = self.input_shape
        self.gradInput = np.zeros((N, C, H, W))

        for n in range(N):
            for c in range(C):
                self.gradInput[n, c, :, :] = gradOutput[n, c] / (H * W)

        return self.gradInput

    def __repr__(self):
        return "GlobalAvgPool2d"

#9. (0.2) Implement [**Flatten**](https://pytorch.org/docs/stable/generated/torch.flatten.html)

In [13]:
class Flatten(Module):
    def __init__(self, start_dim=0, end_dim=-1):
        super(Flatten, self).__init__()

        self.start_dim = start_dim
        self.end_dim = end_dim

    def updateOutput(self, input):
        shape = input.shape
        ndim = len(shape)

        start_dim = self.start_dim if self.start_dim >= 0 else self.start_dim + ndim
        end_dim = self.end_dim if self.end_dim >= 0 else self.end_dim + ndim

        flattened_dim = np.prod(shape[start_dim:end_dim + 1])

        new_shape = (
            shape[:start_dim] +
            (flattened_dim,) +
            shape[end_dim + 1:]
        )

        self.output = input.reshape(new_shape)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput.reshape(input.shape)
        return self.gradInput

    def __repr__(self):
        return "Flatten"

# Activation functions

Here's the complete example for the **Rectified Linear Unit** non-linearity (aka **ReLU**):

In [14]:
class ReLU(Module):
    def __init__(self):
         super(ReLU, self).__init__()

    def updateOutput(self, input):
        self.output = np.maximum(input, 0)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = np.multiply(gradOutput , input > 0)
        return self.gradInput

    def __repr__(self):
        return "ReLU"

## 10. (0.1) Leaky ReLU
Implement [**Leaky Rectified Linear Unit**](http://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29%23Leaky_ReLUs). Expriment with slope.

In [15]:
class LeakyReLU(Module):
    def __init__(self, slope = 0.03):
        super(LeakyReLU, self).__init__()

        self.slope = slope

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.slope * input)
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1, self.slope)
        return self.gradInput

    def __repr__(self):
        return "LeakyReLU"

## 11. (0.1) ELU
Implement [**Exponential Linear Units**](http://arxiv.org/abs/1511.07289) activations.

In [16]:
class ELU(Module):
    def __init__(self, alpha = 1.0):
        super(ELU, self).__init__()

        self.alpha = alpha

    def updateOutput(self, input):
        self.output = np.where(input > 0, input, self.alpha * (np.exp(input) - 1))
        return self.output

    def updateGradInput(self, input, gradOutput):
        self.gradInput = gradOutput * np.where(input > 0, 1, self.alpha * np.exp(input))
        return self.gradInput

    def __repr__(self):
        return "ELU"

## 12. (0.1) SoftPlus
Implement [**SoftPlus**](https://en.wikipedia.org/wiki%2FRectifier_%28neural_networks%29) activations. Look, how they look a lot like ReLU.

In [17]:
class SoftPlus(Module):
    def __init__(self):
        super(SoftPlus, self).__init__()

    def updateOutput(self, input):
        self.output = np.log1p(np.exp(input))
        return self.output

    def updateGradInput(self, input, gradOutput):
        sigmoid = 1.0 / (1.0 + np.exp(-input))
        self.gradInput = gradOutput * sigmoid
        return self.gradInput

    def __repr__(self):
        return "SoftPlus"

#13. (0.2) Gelu
Implement [**Gelu**](https://pytorch.org/docs/stable/generated/torch.nn.GELU.html) activations.

In [18]:
from math import erf

class Gelu(Module):
    def __init__(self):
        super(Gelu, self).__init__()

    def updateOutput(self, input):
        erf_vec = np.vectorize(erf)
        self.output = 0.5 * input * (1.0 + erf_vec(input / np.sqrt(2.0)))
        return self.output

    def updateGradInput(self, input, gradOutput):
        erf_vec = np.vectorize(erf)
        cdf = 0.5 * (1.0 + erf_vec(input / np.sqrt(2.0)))
        pdf = np.exp(-0.5 * input ** 2) / np.sqrt(2.0 * np.pi)
        self.gradInput = gradOutput * (cdf + input * pdf)
        return self.gradInput

    def __repr__(self):
        return "Gelu"

# Criterions

Criterions are used to score the models answers.

In [19]:
class Criterion(object):
    def __init__ (self):
        self.output = None
        self.gradInput = None

    def forward(self, input, target):
        """
            Given an input and a target, compute the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateOutput`.
        """
        return self.updateOutput(input, target)

    def backward(self, input, target):
        """
            Given an input and a target, compute the gradients of the loss function
            associated to the criterion and return the result.

            For consistency this function should not be overrided,
            all the code goes in `updateGradInput`.
        """
        return self.updateGradInput(input, target)

    def updateOutput(self, input, target):
        """
        Function to override.
        """
        return self.output

    def updateGradInput(self, input, target):
        """
        Function to override.
        """
        return self.gradInput

    def __repr__(self):
        """
        Pretty printing. Should be overrided in every module if you want
        to have readable description.
        """
        return "Criterion"

The **MSECriterion**, which is basic L2 norm usually used for regression, is implemented here for you.
- input:   **`batch_size x n_feats`**
- target: **`batch_size x n_feats`**
- output: **scalar**

In [20]:
class MSECriterion(Criterion):
    def __init__(self):
        super(MSECriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = np.sum(np.power(input - target,2)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput  = (input - target) * 2 / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "MSECriterion"

## 14. (0.2) Negative LogLikelihood criterion (numerically unstable)
You task is to implement the **ClassNLLCriterion**. It should implement [multiclass log loss](http://scikit-learn.org/stable/modules/model_evaluation.html#log-loss). Nevertheless there is a sum over `y` (target) in that formula,
remember that targets are one-hot encoded. This fact simplifies the computations a lot. Note, that criterions are the only places, where you divide by batch size. Also there is a small hack with adding small number to probabilities to avoid computing log(0).
- input:   **`batch_size x n_feats`** - probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**



In [21]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15

    def __init__(self):
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        batch_size = input.shape[0]

        if target.ndim == 1:
            self.output = -np.mean(np.log(input_clamp[np.arange(batch_size), target]))
        else:
            self.output = -np.sum(target * np.log(input_clamp)) / batch_size

        return self.output

    def updateGradInput(self, input, target):
        input_clamp = np.clip(input, self.EPS, 1 - self.EPS)
        batch_size = input.shape[0]

        if target.ndim == 1:
            self.gradInput = np.zeros_like(input)
            self.gradInput[np.arange(batch_size), target] = -1.0 / input_clamp[np.arange(batch_size), target]
            self.gradInput /= batch_size
        else:
            self.gradInput = -target / input_clamp / batch_size

        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

## 15. (0.3) Negative LogLikelihood criterion (numerically stable)
- input:   **`batch_size x n_feats`** - log probabilities
- target: **`batch_size x n_feats`** - one-hot representation of ground truth
- output: **scalar**

Task is similar to the previous one, but now the criterion input is the output of log-softmax layer. This decomposition allows us to avoid problems with computation of forward and backward of log().

In [22]:
class ClassNLLCriterionUnstable(Criterion):
    EPS = 1e-15

    def __init__(self):
        super(ClassNLLCriterionUnstable, self).__init__()

    def updateOutput(self, input, target):
        input_clamp = np.clip(input, self.EPS, 1. - self.EPS)
        self.output = -np.sum(target * np.log(input_clamp)) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        input_clamp = np.clip(input, self.EPS, 1. - self.EPS)
        self.gradInput = -target / input_clamp / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterionUnstable"

In [23]:
class ClassNLLCriterion(Criterion):
    def __init__(self):
        super(ClassNLLCriterion, self).__init__()

    def updateOutput(self, input, target):
        self.output = -np.sum(target * input) / input.shape[0]
        return self.output

    def updateGradInput(self, input, target):
        self.gradInput = -target / input.shape[0]
        return self.gradInput

    def __repr__(self):
        return "ClassNLLCriterion"

1-я часть задания: реализация слоев, лосей и функций активации - 5 баллов. \\
2-я часть задания: реализация моделей на своих классах. Что должно быть:
  1. Выберите оптимизатор и реализуйте его, чтоб он работал с вами классами. - 1 балл.
  2. Модель для задачи мультирегрессии на выбраных вами данных. Использовать FCNN, dropout, batchnorm, MSE. Пробуйте различные фукнции активации. Для первой модели попробуйте большую, среднюю и маленькую модель. - 1 балл.
  3. Модель для задачи мультиклассификации на MNIST. Использовать свёртки, макспулы, флэттэны, софтмаксы - 1 балла.
  4. Автоэнкодер для выбранных вами данных. Должен быть на свёртках и полносвязных слоях, дропаутах, батчнормах и тд. - 2 балла. \\

Дополнительно в оценке каждой модели будет учитываться:
1. Наличие правильно выбранной метрики и лосс функции.
2. Отрисовка графиков лосей и метрик на трейне-валидации. Проверка качества модели на тесте.
3. Наличие шедулера для lr.
4. Наличие вормапа.
5. Наличие механизма ранней остановки и сохранение лучшей модели.
6. Свитч лося (метрики) и оптимайзера.

## 2-я часть

In [24]:
import numpy as np
import copy


class SGD:
    def __init__(self, model, lr=1e-2, momentum=0.0, weight_decay=0.0):
        self.model = model
        self.lr = lr
        self.base_lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay

        self.params = self._flatten(model.getParameters())
        self.grads = self._flatten(model.getGradParameters())

        self.velocity = [np.zeros_like(p) for p in self.params]

    def _flatten(self, nested_list):
        flat = []
        for item in nested_list:
            if isinstance(item, list):
                flat.extend(self._flatten(item))
            else:
                flat.append(item)
        return flat

    def zero_grad(self):
        self.model.zeroGradParameters()

    def set_lr(self, lr):
        self.lr = lr

    def step(self):
        for i, (param, grad) in enumerate(zip(self.params, self.grads)):
            if grad is None:
                continue

            grad_to_use = grad
            if self.weight_decay != 0.0:
                grad_to_use = grad_to_use + self.weight_decay * param

            if self.momentum != 0.0:
                self.velocity[i] = self.momentum * self.velocity[i] - self.lr * grad_to_use
                param += self.velocity[i]
            else:
                param -= self.lr * grad_to_use


def iterate_minibatches(X, y=None, batch_size=32, shuffle=True):
    indices = np.arange(len(X))
    if shuffle:
        np.random.shuffle(indices)

    for start_idx in range(0, len(X), batch_size):
        batch_idx = indices[start_idx:start_idx + batch_size]
        if y is None:
            yield X[batch_idx]
        else:
            yield X[batch_idx], y[batch_idx]


def mse_metric(y_pred, y_true):
    return np.mean((y_pred - y_true) ** 2)


def mae_metric(y_pred, y_true):
    return np.mean(np.abs(y_pred - y_true))


def accuracy_metric_from_logits(y_pred, y_true):
    """
    y_pred: logits / log-probs / probs, shape (N, C)
    y_true: one-hot or class indices
    """
    pred_labels = np.argmax(y_pred, axis=1)

    if y_true.ndim == 2:
        true_labels = np.argmax(y_true, axis=1)
    else:
        true_labels = y_true

    return np.mean(pred_labels == true_labels)


def copy_parameters(model):
    params = []

    def _flatten(items):
        flat = []
        for item in items:
            if isinstance(item, list):
                flat.extend(_flatten(item))
            else:
                flat.append(item)
        return flat

    for p in _flatten(model.getParameters()):
        params.append(p.copy())
    return params


def load_parameters(model, saved_params):
    def _flatten(items):
        flat = []
        for item in items:
            if isinstance(item, list):
                flat.extend(_flatten(item))
            else:
                flat.append(item)
        return flat

    current_params = _flatten(model.getParameters())
    for current, saved in zip(current_params, saved_params):
        current[...] = saved


def train_epoch(model, criterion, optimizer, X_train, y_train,
                batch_size=32, metric_fn=None):
    model.train()

    batch_losses = []
    batch_metrics = []

    for X_batch, y_batch in iterate_minibatches(X_train, y_train, batch_size=batch_size, shuffle=True):
        optimizer.zero_grad()

        predictions = model.forward(X_batch)
        loss = criterion.forward(predictions, y_batch)

        grad_loss = criterion.backward(predictions, y_batch)
        model.backward(X_batch, grad_loss)

        optimizer.step()

        batch_losses.append(loss)

        if metric_fn is not None:
            batch_metrics.append(metric_fn(predictions, y_batch))

    epoch_loss = float(np.mean(batch_losses))
    epoch_metric = float(np.mean(batch_metrics)) if metric_fn is not None else None

    return epoch_loss, epoch_metric


def evaluate_epoch(model, criterion, X_val, y_val,
                   batch_size=32, metric_fn=None):
    model.evaluate()

    batch_losses = []
    batch_metrics = []

    for X_batch, y_batch in iterate_minibatches(X_val, y_val, batch_size=batch_size, shuffle=False):
        predictions = model.forward(X_batch)
        loss = criterion.forward(predictions, y_batch)

        batch_losses.append(loss)

        if metric_fn is not None:
            batch_metrics.append(metric_fn(predictions, y_batch))

    epoch_loss = float(np.mean(batch_losses))
    epoch_metric = float(np.mean(batch_metrics)) if metric_fn is not None else None

    return epoch_loss, epoch_metric


def fit_model(model,
              criterion,
              optimizer,
              X_train, y_train,
              X_val=None, y_val=None,
              n_epochs=20,
              batch_size=32,
              metric_fn=None,
              metric_name="metric",
              maximize_metric=False,
              early_stopping_patience=None,
              scheduler_step=None,
              scheduler_gamma=0.1,
              warmup_epochs=0,
              verbose=True):
    """
    scheduler_step:
        если None -> без scheduler
        если int -> каждые scheduler_step эпох lr *= scheduler_gamma
    """

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_metric": [],
        "val_metric": [],
        "lr": []
    }

    best_state = None
    best_epoch = None
    patience_counter = 0

    if X_val is not None and y_val is not None:
        best_score = -np.inf if maximize_metric and metric_fn is not None else np.inf
    else:
        best_score = None

    for epoch in range(1, n_epochs + 1):
        # warmup
        if warmup_epochs > 0 and epoch <= warmup_epochs:
            warmup_lr = optimizer.base_lr * epoch / warmup_epochs
            optimizer.set_lr(warmup_lr)

        train_loss, train_metric = train_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            X_train=X_train,
            y_train=y_train,
            batch_size=batch_size,
            metric_fn=metric_fn
        )

        history["train_loss"].append(train_loss)
        history["train_metric"].append(train_metric)
        history["lr"].append(optimizer.lr)

        if X_val is not None and y_val is not None:
            val_loss, val_metric = evaluate_epoch(
                model=model,
                criterion=criterion,
                X_val=X_val,
                y_val=y_val,
                batch_size=batch_size,
                metric_fn=metric_fn
            )

            history["val_loss"].append(val_loss)
            history["val_metric"].append(val_metric)

            # что считать лучшим
            if metric_fn is not None:
                current_score = val_metric
                improved = current_score > best_score if maximize_metric else current_score < best_score
            else:
                current_score = val_loss
                improved = current_score < best_score

            if improved:
                best_score = current_score
                best_state = copy_parameters(model)
                best_epoch = epoch
                patience_counter = 0
            else:
                patience_counter += 1

            if verbose:
                if metric_fn is not None:
                    print(
                        f"Epoch {epoch:03d} | "
                        f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
                        f"train_{metric_name}={train_metric:.6f} | val_{metric_name}={val_metric:.6f} | "
                        f"lr={optimizer.lr:.6f}"
                    )
                else:
                    print(
                        f"Epoch {epoch:03d} | "
                        f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
                        f"lr={optimizer.lr:.6f}"
                    )
        else:
            if verbose:
                if metric_fn is not None:
                    print(
                        f"Epoch {epoch:03d} | "
                        f"train_loss={train_loss:.6f} | "
                        f"train_{metric_name}={train_metric:.6f} | "
                        f"lr={optimizer.lr:.6f}"
                    )
                else:
                    print(
                        f"Epoch {epoch:03d} | "
                        f"train_loss={train_loss:.6f} | "
                        f"lr={optimizer.lr:.6f}"
                    )

        # scheduler
        if scheduler_step is not None and epoch > warmup_epochs:
            if epoch % scheduler_step == 0:
                optimizer.set_lr(optimizer.lr * scheduler_gamma)

        # early stopping
        if (X_val is not None and y_val is not None and
            early_stopping_patience is not None and
            patience_counter >= early_stopping_patience):
            if verbose:
                print(f"Early stopping at epoch {epoch}")
            break

    # восстановить лучшую модель
    if best_state is not None:
        load_parameters(model, best_state)

    result = {
        "history": history,
        "best_epoch": best_epoch,
        "best_score": best_score
    }

    return result

2-я часть

In [25]:
import numpy as np

class AdamW:
    def __init__(self, model, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        self.model = model
        self.lr = lr
        self.base_lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.weight_decay = weight_decay

        self.params = self._flatten(model.getParameters())
        self.grads  = self._flatten(model.getGradParameters())

        self.m = [np.zeros_like(p) for p in self.params]
        self.v = [np.zeros_like(p) for p in self.params]
        self.t = 0

    def _flatten(self, nested):
        out = []
        for x in nested:
            if isinstance(x, list):
                out.extend(self._flatten(x))
            else:
                out.append(x)
        return out

    def zero_grad(self):
        self.model.zeroGradParameters()

    def set_lr(self, lr):
        self.lr = lr

    def step(self):
        self.t += 1
        b1, b2 = self.beta1, self.beta2

        for i, (p, g) in enumerate(zip(self.params, self.grads)):
            if g is None:
                continue

            # decoupled weight decay
            if self.weight_decay != 0.0:
                p -= self.lr * self.weight_decay * p

            self.m[i] = b1 * self.m[i] + (1 - b1) * g
            self.v[i] = b2 * self.v[i] + (1 - b2) * (g * g)

            m_hat = self.m[i] / (1 - (b1 ** self.t))
            v_hat = self.v[i] / (1 - (b2 ** self.t))

            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
